# CRLA Bronze → Silver → Gold

This notebook traces every transformation applied to the CRLA assessment exports on the way from raw CSV (bronze) to analytical indicators (gold). The goal is an honest audit: what the raw data looks like, what decisions were made during harmonization, and whether the gold-layer metrics are numerically consistent with expectations.

**Pipeline recap**

| Layer | Location | Content |
|---|---|---|
| Bronze | `data/bronze/crla/*.csv` | Raw Looker Studio exports, one file per timepoint |
| Silver | `data/silver/crla/*.parquet` | Harmonized numeric counts + metadata; gitignored |
| Gold | `data/gold/crla_school_timepoints.parquet` | Ordinal mean, SD, skew, excess kurtosis, BC per school × timepoint |
| Gold | `data/gold/crla_school_segments.parquet` | Delta mean/SD/skew + EMD per school × segment |

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Suppress FutureWarnings that don't affect correctness
warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'modules'))

from preprocessing import (
    resolve_latest_exports, load_all_assessments, compute_percentages,
    validate_timepoint, _clean_raw_to_numeric, READING_PROFILES,
    GRADE_LANGUAGE_GROUPS, CANONICAL_GRADE_COLUMNS, METADATA_COLUMNS,
    _get_group_columns, SILVER_CRLA_DIR,
)
from analysis import (
    ORDINAL_WEIGHTS, ALL_TIMEPOINTS, _build_segment_pairs, _segment_label,
    compute_ordinal_moments, compute_emd,
)

print('Imports OK')
print(f'Project root: {PROJECT_ROOT}')

Imports OK
Project root: /workspace/project_crla


---
## 1. Bronze: Raw CSV Inspection

We start by looking at each raw CSV exactly as downloaded from Looker Studio.

In [2]:
file_map = resolve_latest_exports(export_dir=str(PROJECT_ROOT / 'data/bronze/crla'))

print('Files discovered:')
for key, path in sorted(file_map.items()):
    size_mb = Path(path).stat().st_size / 1e6
    print(f'  {key[1]} {key[0]}:  {Path(path).name}  ({size_mb:.1f} MB)')

Files discovered:
  BoSY 2024-25:  CRLA_BoSY_2024-25_20260324_140933.csv  (8.9 MB)
  EoSY 2024-25:  CRLA_EoSY_2024-25_20260324_141017.csv  (9.1 MB)
  BoSY 2025-26:  CRLA_BoSY_2025-26_20260324_140513.csv  (10.1 MB)
  EoSY 2025-26:  CRLA_EoSY_2025-26_20260324_140633.csv  (9.2 MB)
  MoSY 2025-26:  CRLA_MoSY_2025-26_20260324_140555.csv  (9.0 MB)


In [3]:
# Read one raw CSV without any harmonization to see what we're starting with
import csv

key0 = ('2024-25', 'BoSY')
raw_path = file_map[key0]

# First 3 rows, first 15 columns
raw_df_peek = pd.read_csv(raw_path, nrows=3, low_memory=False)
print(f'Shape: {raw_df_peek.shape}')
print(f'\nFirst 15 column names:')
for i, c in enumerate(raw_df_peek.columns[:15]):
    print(f'  [{i:2d}] {repr(c)}')

Shape: (3, 50)

First 15 column names:
  [ 0] 'Region'
  [ 1] 'Division'
  [ 2] 'District'
  [ 3] 'School ID'
  [ 4] 'School Name'
  [ 5] 'Language'
  [ 6] 'Total Assessed'
  [ 7] 'Low Emerging Reader'
  [ 8] 'High Emerging Reader'
  [ 9] 'Developing Reader'
  [10] 'Transitioning Reader'
  [11] 'Reading At Grade Level'
  [12] 'Total Low Emerging'
  [13] 'Total High Emerging'
  [14] 'Total Developing'


In [4]:
# Show all column names for each timepoint — the raw schema differs across school years
for key, path in sorted(file_map.items()):
    df = pd.read_csv(path, nrows=0, low_memory=False)
    grade_cols = [c for c in df.columns if c.startswith('G')]
    non_grade = [c for c in df.columns if not c.startswith('G')]
    print(f"{key[1]} {key[0]}: {len(df.columns)} total cols | {len(non_grade)} non-grade | {len(grade_cols)} grade cols")
    # Show any suspicious column names
    issues = [c for c in df.columns if 'FIl' in c or 'Total' in c]
    if issues:
        print(f'  Notable: {issues[:4]}')

BoSY 2024-25: 50 total cols | 17 non-grade | 33 grade cols
  Notable: ['Total Assessed', 'Total Low Emerging', 'Total High Emerging', 'Total Developing']
EoSY 2024-25: 50 total cols | 17 non-grade | 33 grade cols
  Notable: ['Total Assessed', 'Total Low Emerging', 'Total High Emerging', 'Total Developing']
BoSY 2025-26: 53 total cols | 17 non-grade | 36 grade cols
  Notable: ['Total Assessed', 'Total Low Emerging', 'Total High Emerging', 'Total Developing']
EoSY 2025-26: 47 total cols | 17 non-grade | 30 grade cols
  Notable: ['Total Assessed', 'Total Low Emerging', 'Total High Emerging', 'Total Developing']
MoSY 2025-26: 47 total cols | 17 non-grade | 30 grade cols
  Notable: ['Total Assessed', 'Total Low Emerging', 'Total High Emerging', 'Total Developing']


---
## 2. Harmonization: What `harmonize_columns()` Actually Does

Four operations run in sequence. We show before/after for each.

In [5]:
# Operation 1: FIl → Fil typo fix
# This was a source-data inconsistency in some exports.
raw_df_2024_bosy = pd.read_csv(file_map[('2024-25','BoSY')], nrows=5, low_memory=False)

fil_typos = [c for c in raw_df_2024_bosy.columns if 'FIl' in c]
print(f'Columns containing "FIl" typo in BoSY 2024-25: {len(fil_typos)}')
print(f'Examples: {fil_typos[:4]}')
print(f'After fix: {[c.replace("FIl","Fil") for c in fil_typos[:4]]}')

Columns containing "FIl" typo in BoSY 2024-25: 1
Examples: ['G2 FIl Higher Emergent']
After fix: ['G2 Fil Higher Emergent']


In [6]:
# Operation 2: Mojibake fix (¤ → ñ, Ã± → ñ)
# Appears in school/division names containing ñ (e.g., Dueñas, Osmeña, Doña)
raw_df = pd.read_csv(file_map[('2024-25','BoSY')], low_memory=False)

# Find examples of corrupted ñ
corrupted_a4 = raw_df['School Name'].str.contains('\u00a4', na=False)
corrupted_a3 = raw_df['School Name'].str.contains('Ã±', na=False)
print(f'School names with ¤ (U+00A4 corruption): {corrupted_a4.sum()}')
print(f'School names with Ã± (double-encoded):     {corrupted_a3.sum()}')
if corrupted_a4.sum() > 0:
    sample = raw_df.loc[corrupted_a4, 'School Name'].head(3).tolist()
    print(f'Samples (raw): {sample}')
    print(f'After fix: {[s.replace(chr(0xa4), "ñ") for s in sample]}')

School names with ¤ (U+00A4 corruption): 2
School names with Ã± (double-encoded):     1
Samples (raw): ['Buenavista ES (Due¤as)', 'Osme¤a ES']
After fix: ['Buenavista ES (Dueñas)', 'Osmeña ES']


In [7]:
# Operation 3: Total columns dropped
# Source data has per-grade totals like "G1 Total Assessed", "G2 Total MT Assessed".
# These vary in naming across school years and are redundant (derivable from profile counts).
total_cols = [c for c in raw_df.columns if c.startswith('G') and 'total' in c.lower()]
print(f'Grade-level Total columns in BoSY 2024-25 ({len(total_cols)}):')
print(total_cols)

Grade-level Total columns in BoSY 2024-25 (3):
['G1 Total Assessed', 'G2 Total Assessed', 'G3 Total Assessed']


In [8]:
# Operation 4: Missing canonical columns filled with NaN
# EoSY and MoSY 2025-26 are missing G3 MT columns (the mother tongue assessment for
# G3 was apparently not conducted). The pipeline fills these with NaN so downstream
# code doesn't have to check for missing columns.
from preprocessing import harmonize_columns, CANONICAL_GRADE_COLUMNS

print('Canonical columns (30 total):')
for i, c in enumerate(CANONICAL_GRADE_COLUMNS):
    print(f'  [{i+1:2d}] {c}')

Canonical columns (30 total):
  [ 1] G1 Lower Emergent
  [ 2] G1 Higher Emergent
  [ 3] G1 Developing
  [ 4] G1 Transitioning
  [ 5] G1 Grade Level
  [ 6] G2 MT Lower Emergent
  [ 7] G2 MT Higher Emergent
  [ 8] G2 MT Developing
  [ 9] G2 MT Transitioning
  [10] G2 MT Grade Level
  [11] G2 Fil Lower Emergent
  [12] G2 Fil Higher Emergent
  [13] G2 Fil Developing
  [14] G2 Fil Transitioning
  [15] G2 Fil Grade Level
  [16] G3 MT Lower Emergent
  [17] G3 MT Higher Emergent
  [18] G3 MT Developing
  [19] G3 MT Transitioning
  [20] G3 MT Grade Level
  [21] G3 Fil Lower Emergent
  [22] G3 Fil Higher Emergent
  [23] G3 Fil Developing
  [24] G3 Fil Transitioning
  [25] G3 Fil Grade Level
  [26] G3 Eng Lower Emergent
  [27] G3 Eng Higher Emergent
  [28] G3 Eng Developing
  [29] G3 Eng Transitioning
  [30] G3 Eng Grade Level


In [9]:
# Which canonical columns are NaN in each timepoint? (expected: G3 MT in EoSY/MoSY 2025-26)
print('Missing canonical columns per timepoint (NaN-filled by harmonize_columns):\n')
for key, path in sorted(file_map.items()):
    df_raw = pd.read_csv(path, nrows=5, low_memory=False)
    df_harm = harmonize_columns(df_raw)
    grade_in_harm = [c for c in df_harm.columns if c in CANONICAL_GRADE_COLUMNS]
    nan_cols = [c for c in CANONICAL_GRADE_COLUMNS if c not in grade_in_harm or df_harm[c].isna().all()]
    if nan_cols:
        print(f'  {key[1]} {key[0]}: {nan_cols}')
    else:
        print(f'  {key[1]} {key[0]}: all 30 columns present')

Missing canonical columns per timepoint (NaN-filled by harmonize_columns):

  BoSY 2024-25: all 30 columns present
  EoSY 2024-25: all 30 columns present
  BoSY 2025-26: all 30 columns present
  ⚠ Filled 5 missing grade columns with NaN: ['G3 MT Developing', 'G3 MT Grade Level', 'G3 MT Higher Emergent', 'G3 MT Lower Emergent', 'G3 MT Transitioning']
  EoSY 2025-26: ['G3 MT Lower Emergent', 'G3 MT Higher Emergent', 'G3 MT Developing', 'G3 MT Transitioning', 'G3 MT Grade Level']
  ⚠ Filled 5 missing grade columns with NaN: ['G3 MT Developing', 'G3 MT Grade Level', 'G3 MT Higher Emergent', 'G3 MT Lower Emergent', 'G3 MT Transitioning']
  MoSY 2025-26: ['G3 MT Lower Emergent', 'G3 MT Higher Emergent', 'G3 MT Developing', 'G3 MT Transitioning', 'G3 MT Grade Level']


---
## 3. Silver Layer Verification

The silver parquets store harmonized raw counts. Here we verify:
- School counts match bronze
- Numeric types are correct (no strings, no comma-formatted numbers)
- `total_assessed` is populated from the dashboard's `Total Assessed` column

In [10]:
silver_dir = PROJECT_ROOT / 'data/silver/crla'

rows = []
for key in sorted(file_map.keys()):
    sy, period = key
    fpath = silver_dir / f'{sy}_{period}.parquet'
    if not fpath.exists():
        print(f'MISSING: {fpath}')
        continue
    df = pd.read_parquet(fpath)
    rows.append({
        'Timepoint': f'{period} {sy}',
        'Schools': len(df),
        'Columns': len(df.columns),
        'dtype_grade_cols': str(df[CANONICAL_GRADE_COLUMNS[0]].dtype),
        'total_assessed_present': 'total_assessed' in df.columns,
        'total_assessed_nulls': df['total_assessed'].isna().sum() if 'total_assessed' in df.columns else 'N/A',
    })

pd.DataFrame(rows).set_index('Timepoint')

,Schools,Columns,dtype_grade_cols,total_assessed_present,total_assessed_nulls
Timepoint,,,,,
BoSY 2024-25,35280,49,int64,True,0
EoSY 2024-25,37045,49,int64,True,0
BoSY 2025-26,38983,49,int64,True,0
EoSY 2025-26,38743,49,int64,True,0
MoSY 2025-26,38297,49,int64,True,0


In [11]:
# Spot-check: compare a silver row against the original bronze row for the same school
df_silver = pd.read_parquet(silver_dir / '2024-25_BoSY.parquet')
df_bronze = pd.read_csv(file_map[('2024-25','BoSY')], low_memory=False)

# Find first school present in both
bronze_ids = pd.to_numeric(df_bronze['School ID'], errors='coerce').dropna().astype(int)
common_id = df_silver.index.intersection(bronze_ids)[0]

bronze_row = df_bronze[df_bronze['School ID'].astype(str) == str(common_id)].iloc[0]
silver_row = df_silver.loc[common_id]

print(f'School ID: {common_id}')
print(f'School Name (silver): {silver_row["School Name"]}')
print()
print('G1 profile columns (raw bronze vs silver):')
for col in _get_group_columns('G1', None):
    b_val = bronze_row.get(col, 'N/A')
    s_val = silver_row.get(col, 'N/A')
    print(f'  {col:35s}  bronze: {str(b_val):>8}   silver: {s_val}')

School ID: 131240
School Name (silver): Cotabato City Central Pilot School

G1 profile columns (raw bronze vs silver):
  G1 Lower Emergent                    bronze:      646   silver: 646
  G1 Higher Emergent                   bronze:       66   silver: 66
  G1 Developing                        bronze:      144   silver: 144
  G1 Transitioning                     bronze:       57   silver: 57
  G1 Grade Level                       bronze:        9   silver: 9


---
## 4. Percentage Computation

`compute_percentages()` divides each profile count by the row-sum within its grade-language group. We verify that percentages sum to 100 for all non-NaN groups.

In [12]:
# Load all assessments and compute percentages
df_all = load_all_assessments(file_map=file_map, source='local')
percentages = {}
for key, df in df_all.items():
    percentages[key] = compute_percentages(df)
print('Percentages computed for all timepoints.')

Loading 2024-25 BoSY from /workspace/project_crla/data/bronze/crla/CRLA_BoSY_2024-25_20260324_140933.csv ...


  → 35280 schools loaded.
Loading 2025-26 BoSY from /workspace/project_crla/data/bronze/crla/CRLA_BoSY_2025-26_20260324_140513.csv ...


  → 38983 schools loaded.
Loading 2024-25 EoSY from /workspace/project_crla/data/bronze/crla/CRLA_EoSY_2024-25_20260324_141017.csv ...


  → 37045 schools loaded.
Loading 2025-26 EoSY from /workspace/project_crla/data/bronze/crla/CRLA_EoSY_2025-26_20260324_140633.csv ...


  ⚠ Filled 5 missing grade columns with NaN: ['G3 MT Developing', 'G3 MT Grade Level', 'G3 MT Higher Emergent', 'G3 MT Lower Emergent', 'G3 MT Transitioning']
  → 38743 schools loaded.
Loading 2025-26 MoSY from /workspace/project_crla/data/bronze/crla/CRLA_MoSY_2025-26_20260324_140555.csv ...


  ⚠ Filled 5 missing grade columns with NaN: ['G3 MT Developing', 'G3 MT Grade Level', 'G3 MT Higher Emergent', 'G3 MT Lower Emergent', 'G3 MT Transitioning']
  → 38297 schools loaded.


Percentages computed for all timepoints.


In [13]:
# Verify each group sums to 100 (within floating-point tolerance)
print('Group percentage sums — should all be ≈ 100 or NaN\n')

key = ('2024-25', 'BoSY')
pct_df = percentages[key]

for grade, lang in GRADE_LANGUAGE_GROUPS:
    cols = _get_group_columns(grade, lang)
    row_sums = pct_df[cols].sum(axis=1)
    non_nan = row_sums.dropna()
    # Schools with full data should sum to exactly 100
    all_100 = np.allclose(non_nan, 100, atol=0.01)
    gl_label = f'{grade} {lang}' if lang else grade
    print(f'  {gl_label:12s}: {len(non_nan):5d} schools with data | sums ≈ 100: {all_100} | '
          f'min={non_nan.min():.2f} max={non_nan.max():.2f}')

Group percentage sums — should all be ≈ 100 or NaN

  G1          : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00
  G2 MT       : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00
  G2 Fil      : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00
  G3 MT       : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00
  G3 Fil      : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00
  G3 Eng      : 35280 schools with data | sums ≈ 100: False | min=0.00 max=100.00


In [14]:
# Show a sample school's profile breakdown
print(f'Percentage breakdown for School {common_id} ({key[1]} {key[0]})\n')
pct_row = pct_df.loc[common_id]

for grade, lang in GRADE_LANGUAGE_GROUPS:
    cols = _get_group_columns(grade, lang)
    gl_label = f'{grade} {lang}' if lang else grade
    vals = pct_row[cols]
    if vals.isna().all():
        print(f'  {gl_label:12s}: (no data)')
    else:
        bar = ''.join(['█' * int(v/5) for v in vals])
        print(f'  {gl_label:12s}: ' + ' | '.join(f'{p:5.1f}%' for p in vals) + f'  (LE→GL)')

Percentage breakdown for School 131240 (BoSY 2024-25)

  G1          :  70.1% |   7.2% |  15.6% |   6.2% |   1.0%  (LE→GL)
  G2 MT       :  26.0% |   6.8% |   8.1% |  51.9% |   7.2%  (LE→GL)
  G2 Fil      :  26.0% |   6.8% |   8.1% |  51.9% |   7.2%  (LE→GL)
  G3 MT       :  20.3% |   2.0% |  10.4% |  60.5% |   6.9%  (LE→GL)
  G3 Fil      :  20.3% |   2.0% |  10.4% |  60.5% |   6.9%  (LE→GL)
  G3 Eng      :  11.4% |  13.8% |  14.6% |  49.7% |  10.6%  (LE→GL)


---
## 5. Validation: `valid` vs `valid_strict`

`valid` = at least one grade-language group has data.  
`valid_strict` = all three grades present + ≥4 groups + ≥15 assessed per group.

This is the gate for priority ranking.

In [15]:
validation = {}
for key, df in df_all.items():
    validation[key] = validate_timepoint(
        percentages[key],
        raw_counts_df=_clean_raw_to_numeric(df),
    )

rows = []
for key in sorted(df_all.keys()):
    val = validation[key]
    total = len(val)
    n_valid = val['valid'].sum()
    n_strict = val['valid_strict'].sum()
    rows.append({
        'Timepoint': f"{key[1]} {key[0]}",
        'Total Schools': total,
        'valid': n_valid,
        'valid %': f'{n_valid/total*100:.1f}%',
        'valid_strict': n_strict,
        'strict %': f'{n_strict/total*100:.1f}%',
    })

pd.DataFrame(rows).set_index('Timepoint')

,Total Schools,valid,valid %,valid_strict,strict %
Timepoint,,,,,
BoSY 2024-25,35280,35280,100.0%,22961,65.1%
EoSY 2024-25,37045,37042,100.0%,24380,65.8%
BoSY 2025-26,38983,38982,100.0%,24304,62.3%
EoSY 2025-26,38743,38743,100.0%,23299,60.1%
MoSY 2025-26,38297,38297,100.0%,14182,37.0%


In [16]:
# Break down WHY schools fail strict validation (BoSY 2024-25 as example)
key = ('2024-25', 'BoSY')
val = validation[key]
failed = val[val['valid'] & ~val['valid_strict']]

print(f'Schools that are valid but NOT valid_strict in BoSY 2024-25: {len(failed):,}\n')
print('Failure breakdown:')

has_cols = [c for c in val.columns if c.startswith('has_')]

# Not all grades
missing_grade = ~val['has_all_grades'] & val['valid']
print(f'  Missing at least one grade level: {missing_grade.sum():,}')

# Too few groups
few_groups = (val['groups_available'] < 4) & val['valid'] & val['has_all_grades']
print(f'  All grades present but < 4 groups: {few_groups.sum():,}')

# Too few per group
too_small = (val['min_group_assessed'] < 15) & val['valid'] & val['has_all_grades'] & (val['groups_available'] >= 4)
print(f'  Enough groups but < 15 per group: {too_small.sum():,}')

Schools that are valid but NOT valid_strict in BoSY 2024-25: 12,319

Failure breakdown:
  Missing at least one grade level: 411
  All grades present but < 4 groups: 56
  Enough groups but < 15 per group: 11,852


In [17]:
# Distribution of min_group_assessed for valid schools
fig_data = val[val['valid']]['min_group_assessed'].dropna()
print(f'min_group_assessed distribution (valid schools, BoSY 2024-25):')
print(f'  Min: {fig_data.min():.0f}   Median: {fig_data.median():.0f}   Mean: {fig_data.mean():.1f}   Max: {fig_data.max():.0f}')

# Quantiles
qs = [0, 5, 10, 15, 25, 50, 75, 100]
for q in qs:
    print(f'  {q:3d}th pctile: {fig_data.quantile(q/100):.0f}')

min_group_assessed distribution (valid schools, BoSY 2024-25):
  Min: 1   Median: 21   Mean: 43.2   Max: 1680
    0th pctile: 1
    5th pctile: 4
   10th pctile: 6
   15th pctile: 8
   25th pctile: 11
   50th pctile: 21
   75th pctile: 42
  100th pctile: 1680


---
## 6. Ordinal Moments: Five Statistics per School

`compute_ordinal_moments()` computes five distributional statistics from the 5-point ordinal profile.

| Statistic | Formula | Interpretation |
|---|---|---|
| `ordinal_mean` | Σ(p_i × w_i) / 100 | Average proficiency level, scale 1–5 |
| `ordinal_sd` | √(Σ p_i (w_i − mean)²/100) | Spread across profiles |
| `ordinal_skew` | third central moment / sd³ | Negative = concentrated at high end |
| `ordinal_kurt` | fourth central moment / sd⁴ − 3 | Excess kurtosis; positive = more peaked |
| `bimodality_coef` | (skew² + 1) / regular_kurtosis | BC > 0.555 flags bimodal character |

In [18]:
# Compute moments for all timepoints
print('National mean moments per timepoint (valid schools only):\n')
rows = []
for key in sorted(df_all.keys()):
    pct = percentages[key]
    val = validation[key]
    moments = compute_ordinal_moments(pct)
    valid_mask = val['valid']
    m = moments[valid_mask]
    rows.append({
        'Timepoint': f"{key[1]} {key[0]}",
        'N valid': valid_mask.sum(),
        'mean': m['ordinal_mean'].mean(),
        'sd': m['ordinal_sd'].mean(),
        'skew': m['ordinal_skew'].mean(),
        'exc. kurt': m['ordinal_kurt'].mean(),
        'BC': m['bimodality_coef'].mean(),
    })

pd.DataFrame(rows).set_index('Timepoint').round(3)

National mean moments per timepoint (valid schools only):



,N valid,mean,sd,skew,exc. kurt,BC
Timepoint,,,,,,
BoSY 2024-25,35280,3.303,1.071,-0.492,0.242,0.761
EoSY 2024-25,37042,4.127,0.872,-1.179,1.651,0.779
BoSY 2025-26,38982,3.060,1.076,-0.314,0.193,0.780
EoSY 2025-26,38743,4.147,0.827,-1.127,1.512,0.778
MoSY 2025-26,38297,3.228,0.962,-0.408,-0.139,0.778


In [19]:
# Verify a school's moments by hand for BoSY 2024-25
w = np.array([1, 2, 3, 4, 5], dtype=float)

# Pick a school with all 6 groups
pct_bosy = percentages[('2024-25', 'BoSY')]
moments_bosy = compute_ordinal_moments(pct_bosy)

# Find school with all groups
full_groups = validation[('2024-25','BoSY')]['groups_available'] == 6
check_id = validation[('2024-25','BoSY')][full_groups].index[0]

print(f'Manual moment check for School ID {check_id}:\n')
for grade, lang in GRADE_LANGUAGE_GROUPS:
    cols = _get_group_columns(grade, lang)
    p = pct_bosy.loc[check_id, cols].values.astype(float) / 100
    if np.any(np.isnan(p)):
        continue
    mean_ = (p * w).sum()
    dev = w - mean_
    sd_ = np.sqrt((p * dev**2).sum())
    skew_ = (p * dev**3).sum() / sd_**3
    kurt_reg = (p * dev**4).sum() / sd_**4
    bc_ = (skew_**2 + 1) / kurt_reg
    gl_label = f'{grade} {lang}' if lang else grade
    print(f'  {gl_label:12s}  mean={mean_:.3f}  sd={sd_:.3f}  skew={skew_:.3f}  kurt_exc={kurt_reg-3:.3f}  BC={bc_:.3f}')

print(f'\n  Overall (from compute_ordinal_moments):')
row = moments_bosy.loc[check_id]
print(f'  mean={row["ordinal_mean"]:.3f}  sd={row["ordinal_sd"]:.3f}  skew={row["ordinal_skew"]:.3f}  kurt_exc={row["ordinal_kurt"]:.3f}  BC={row["bimodality_coef"]:.3f}')

Manual moment check for School ID 131240:

  G1            mean=1.608  sd=1.019  skew=1.411  kurt_exc=0.705  BC=0.807
  G2 MT         mean=3.075  sd=1.382  skew=-0.563  kurt_exc=-1.278  BC=0.765
  G2 Fil        mean=3.075  sd=1.382  skew=-0.563  kurt_exc=-1.278  BC=0.765
  G3 MT         mean=3.317  sd=1.269  skew=-1.003  kurt_exc=-0.462  BC=0.790
  G3 Fil        mean=3.317  sd=1.269  skew=-1.003  kurt_exc=-0.462  BC=0.790
  G3 Eng        mean=3.344  sd=1.181  skew=-0.717  kurt_exc=-0.546  BC=0.617

  Overall (from compute_ordinal_moments):
  mean=2.956  sd=1.250  skew=-0.406  kurt_exc=-0.553  BC=0.756


In [20]:
# BC interpretation: what does BC ≈ 0.76 mean for our scale?
# Reference points:
print('BC reference values for our 5-point ordinal scale:\n')

# Perfect bimodal: 50% at 1, 50% at 5
p_bimodal = np.array([0.5, 0, 0, 0, 0.5])
mean_b = (p_bimodal * w).sum()
dev_b = w - mean_b
sd_b = np.sqrt((p_bimodal * dev_b**2).sum())
skew_b = (p_bimodal * dev_b**3).sum() / sd_b**3
kurt_b = (p_bimodal * dev_b**4).sum() / sd_b**4
bc_b = (skew_b**2 + 1) / kurt_b
print(f'Perfect bimodal (50% LE, 50% GL):  mean={mean_b:.2f}  BC={bc_b:.3f}')

# Uniform: equal weight across all 5
p_uniform = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
mean_u = (p_uniform * w).sum()
dev_u = w - mean_u
sd_u = np.sqrt((p_uniform * dev_u**2).sum())
skew_u = (p_uniform * dev_u**3).sum() / sd_u**3
kurt_u = (p_uniform * dev_u**4).sum() / sd_u**4
bc_u = (skew_u**2 + 1) / kurt_u
print(f'Uniform (20% each):                mean={mean_u:.2f}  BC={bc_u:.3f}')

# Concentrated at GL
p_gl = np.array([0.02, 0.03, 0.05, 0.20, 0.70])
mean_g = (p_gl * w).sum()
dev_g = w - mean_g
sd_g = np.sqrt((p_gl * dev_g**2).sum())
skew_g = (p_gl * dev_g**3).sum() / sd_g**3
kurt_g = (p_gl * dev_g**4).sum() / sd_g**4
bc_g = (skew_g**2 + 1) / kurt_g
print(f'Concentrated at GL (70% GL):       mean={mean_g:.2f}  BC={bc_g:.3f}')

# BoSY-like: spread across lower profiles
p_bosy = np.array([0.18, 0.22, 0.25, 0.20, 0.15])
mean_bo = (p_bosy * w).sum()
dev_bo = w - mean_bo
sd_bo = np.sqrt((p_bosy * dev_bo**2).sum())
skew_bo = (p_bosy * dev_bo**3).sum() / sd_bo**3
kurt_bo = (p_bosy * dev_bo**4).sum() / sd_bo**4
bc_bo = (skew_bo**2 + 1) / kurt_bo
print(f'BoSY-like (spread, slight low):    mean={mean_bo:.2f}  BC={bc_bo:.3f}')

print(f'\nConventional bimodality threshold: BC > 0.555')
print('Note: for a 5-category ordinal scale, the uniform distribution itself has BC > 0.555.')
print('Use BC as a RELATIVE indicator — higher BC = more bimodal character relative to other schools.')

BC reference values for our 5-point ordinal scale:

Perfect bimodal (50% LE, 50% GL):  mean=3.00  BC=1.000
Uniform (20% each):                mean=3.00  BC=0.588
Concentrated at GL (70% GL):       mean=4.53  BC=0.756
BoSY-like (spread, slight low):    mean=2.92  BC=0.531

Conventional bimodality threshold: BC > 0.555
Note: for a 5-category ordinal scale, the uniform distribution itself has BC > 0.555.
Use BC as a RELATIVE indicator — higher BC = more bimodal character relative to other schools.


---
## 7. EMD: Earth Mover's Distance Between Timepoints

`compute_emd(pct_t0, pct_t1)` uses scipy's `wasserstein_distance` to measure how much the profile distribution *shifted* between two timepoints. Unlike delta_mean, EMD is sensitive to distributional rearrangements even when the mean is unchanged.

In [21]:
# Compute EMD for the Learning 2024-25 segment
t0 = ('2024-25', 'BoSY')
t1 = ('2024-25', 'EoSY')

emd_series = compute_emd(percentages[t0], percentages[t1])
valid_emd = emd_series.dropna()

print(f'EMD for Learning 2024-25 (BoSY → EoSY):')
print(f'  Schools with EMD: {len(valid_emd):,}')
print(f'  Mean EMD: {valid_emd.mean():.3f}')
print(f'  Median:   {valid_emd.median():.3f}')
print(f'  SD:       {valid_emd.std():.3f}')
print(f'  Min:      {valid_emd.min():.3f}  Max: {valid_emd.max():.3f}')

EMD for Learning 2024-25 (BoSY → EoSY):
  Schools with EMD: 35,140
  Mean EMD: 0.906
  Median:   0.846
  SD:       0.406
  Min:      0.000  Max: 4.000


In [22]:
# Compare EMD vs delta_mean: do they tell the same story?
moments0 = compute_ordinal_moments(percentages[t0])
moments1 = compute_ordinal_moments(percentages[t1])
val0 = validation[t0]
val1 = validation[t1]

both_valid = val0['valid'] & val1['valid']
common = moments0.index.intersection(moments1.index)
both_valid_common = both_valid.reindex(common, fill_value=False)

delta_mean = (moments1['ordinal_mean'] - moments0['ordinal_mean']).reindex(common)
emd_common = emd_series.reindex(common)

valid_mask = both_valid_common
corr = delta_mean[valid_mask].corr(emd_common[valid_mask])
print(f'Correlation between delta_mean and EMD (valid schools): {corr:.3f}')

# Schools with near-zero delta_mean but non-trivial EMD
# (hidden distributional redistribution)
low_delta = valid_mask & (delta_mean.abs() < 0.1)
high_emd = valid_mask & (emd_common > emd_common[valid_mask].quantile(0.75))
nontrivial = low_delta & high_emd
print(f'\nSchools with |delta_mean| < 0.1 but EMD in top quartile: {nontrivial.sum()}')
print('(These show internal reshuffling that delta_mean misses)')

Correlation between delta_mean and EMD (valid schools): 0.879

Schools with |delta_mean| < 0.1 but EMD in top quartile: 14
(These show internal reshuffling that delta_mean misses)


In [23]:
# Look at one such school: what does its profile look like at both ends?
if nontrivial.sum() > 0:
    example_id = nontrivial[nontrivial].index[0]
    print(f'Example school ID {example_id}: delta_mean={delta_mean[example_id]:.3f}, EMD={emd_common[example_id]:.3f}\n')
    print('BoSY profile (G1, all groups averaged):')
    for grade, lang in GRADE_LANGUAGE_GROUPS[:3]:
        cols = _get_group_columns(grade, lang)
        gl_label = f'{grade} {lang}' if lang else grade
        p0 = percentages[t0].reindex([example_id])[cols].values.flatten()
        p1 = percentages[t1].reindex([example_id])[cols].values.flatten()
        if np.any(np.isnan(p0)) or np.any(np.isnan(p1)):
            continue
        print(f'  {gl_label:12s}  BoSY: {" | ".join(f"{v:5.1f}" for v in p0)}')
        print(f'  {"":12s}  EoSY: {" | ".join(f"{v:5.1f}" for v in p1)}')
        print(f'  {"":12s}  Labels: LE | HE | Dev | Trans | GL')
        print()

Example school ID 135598: delta_mean=0.069, EMD=1.264

BoSY profile (G1, all groups averaged):
  G1            BoSY: 100.0 |   0.0 |   0.0 |   0.0 |   0.0
                EoSY:   0.0 |   0.0 |   0.0 | 100.0 |   0.0
                Labels: LE | HE | Dev | Trans | GL

  G2 MT         BoSY:  25.0 |   0.0 |  50.0 |   0.0 |  25.0
                EoSY:   0.0 |   0.0 |  50.0 |  50.0 |   0.0
                Labels: LE | HE | Dev | Trans | GL

  G2 Fil        BoSY:   0.0 |  25.0 |   0.0 |   0.0 |  75.0
                EoSY:   0.0 |   0.0 |  50.0 |  50.0 |   0.0
                Labels: LE | HE | Dev | Trans | GL



---
## 8. Gold Layer Inspection

The two gold parquets are the outputs consumed by downstream tools.

In [24]:
gold_tp = pd.read_parquet(PROJECT_ROOT / 'data/gold/crla_school_timepoints.parquet')
gold_seg = pd.read_parquet(PROJECT_ROOT / 'data/gold/crla_school_segments.parquet')

print('crla_school_timepoints.parquet')
print(f'  Shape: {gold_tp.shape}')
print(f'  Columns: {list(gold_tp.columns)}')
print()
print('crla_school_segments.parquet')
print(f'  Shape: {gold_seg.shape}')
print(f'  Columns: {list(gold_seg.columns)}')

crla_school_timepoints.parquet
  Shape: (188348, 26)
  Columns: ['School ID', 'school_year', 'period', 'timepoint_label', 'total_assessed', 'ordinal_overall', 'ordinal_G1', 'ordinal_G2', 'ordinal_G3', 'pct_gl', 'ordinal_mean', 'ordinal_sd', 'ordinal_skew', 'ordinal_kurt', 'bimodality_coef', 'valid', 'valid_strict', 'School Name', 'Region', 'Division', 'District', 'ordinal_G2_MT', 'ordinal_G2_Fil', 'ordinal_G3_MT', 'ordinal_G3_Fil', 'ordinal_G3_Eng']

crla_school_segments.parquet
  Shape: (259148, 12)
  Columns: ['School ID', 'tp_from', 'tp_to', 'segment_label', 'seg_idx', 'delta_mean', 'delta_sd', 'delta_skew', 'emd_mean', 'valid', 'valid_strict', 'count_stable']


In [25]:
# National means per timepoint — cross-check against known benchmarks
print('National averages by timepoint (valid schools only):\n')
rows = []
for label in gold_tp['timepoint_label'].unique():
    sub = gold_tp[(gold_tp['timepoint_label'] == label) & gold_tp['valid']]
    rows.append({
        'Timepoint': label,
        'N valid': len(sub),
        'ordinal_mean': sub['ordinal_mean'].mean(),
        'ordinal_sd': sub['ordinal_sd'].mean(),
        'ordinal_skew': sub['ordinal_skew'].mean(),
        'ordinal_kurt': sub['ordinal_kurt'].mean(),
        'BC': sub['bimodality_coef'].mean(),
    })
    
df_summary = pd.DataFrame(rows).set_index('Timepoint').sort_values('ordinal_mean')
print(df_summary.round(3).to_string())

print('\nExpected (from memory):')
print('  BoSY 2024-25: mean≈3.30, sd≈1.07')
print('  EoSY 2024-25: mean≈4.13, sd≈0.87')
print('  BoSY 2025-26: mean≈3.06, sd≈1.08')
print('  EoSY 2025-26: mean≈4.15')

National averages by timepoint (valid schools only):

              N valid  ordinal_mean  ordinal_sd  ordinal_skew  ordinal_kurt     BC
Timepoint                                                                         
BoSY 2025-26    38982         3.060       1.076        -0.314         0.193  0.780
MoSY 2025-26    38297         3.228       0.962        -0.408        -0.139  0.778
BoSY 2024-25    35280         3.303       1.071        -0.492         0.242  0.761
EoSY 2024-25    37042         4.127       0.872        -1.179         1.651  0.779
EoSY 2025-26    38743         4.147       0.827        -1.127         1.512  0.778

Expected (from memory):
  BoSY 2024-25: mean≈3.30, sd≈1.07
  EoSY 2024-25: mean≈4.13, sd≈0.87
  BoSY 2025-26: mean≈3.06, sd≈1.08
  EoSY 2025-26: mean≈4.15


In [26]:
# Segment deltas — cross-check against known results
print('Mean segment metrics by segment label (valid schools only):\n')
rows = []
for label in gold_seg['segment_label'].unique():
    sub = gold_seg[(gold_seg['segment_label'] == label) & gold_seg['valid']]
    rows.append({
        'Segment': label,
        'N valid': len(sub),
        'delta_mean': sub['delta_mean'].mean(),
        'delta_sd': sub['delta_sd'].mean(),
        'delta_skew': sub['delta_skew'].mean(),
        'emd_mean': sub['emd_mean'].mean(),
    })

df_segs = pd.DataFrame(rows).set_index('Segment')
print(df_segs.round(3).to_string())

print('\nExpected: Learning 2024-25 delta≈+0.82, Learning 2025-26 delta≈+1.10')

Mean segment metrics by segment label (valid schools only):

                             N valid  delta_mean  delta_sd  delta_skew  emd_mean
Segment                                                                         
Learning_2024-25               34576       0.817    -0.200      -0.675     0.903
BoSYMoSY_2025-26               19025       0.541    -0.036      -0.374     0.743
MoSYEoSY_2025-26               18981       0.562    -0.205      -0.456     0.634
Learning_2025-26               35968       1.093    -0.251      -0.823     1.222
YoY_BoSY_2024-25_to_2025-26    31325      -0.221     0.003       0.156     0.675
YoY_EoSY_2024-25_to_2025-26    32681       0.053    -0.048       0.016     0.502
EndToEnd_2024-25_to_2025-26    30955       0.874    -0.251      -0.672     1.080

Expected: Learning 2024-25 delta≈+0.82, Learning 2025-26 delta≈+1.10


In [27]:
# NaN coverage in gold — how many schools have each metric
print('NaN rates in crla_school_timepoints.parquet (valid schools only):\n')
valid_tp = gold_tp[gold_tp['valid']]
metric_cols = ['ordinal_mean','ordinal_sd','ordinal_skew','ordinal_kurt','bimodality_coef','pct_gl']
for col in metric_cols:
    n_nan = valid_tp[col].isna().sum()
    print(f'  {col:25s}: {n_nan:5d} NaN ({n_nan/len(valid_tp)*100:.1f}%)')

print()
print('NaN rates in crla_school_segments.parquet (valid segments only):\n')
valid_seg = gold_seg[gold_seg['valid']]
for col in ['delta_mean','delta_sd','delta_skew','emd_mean']:
    n_nan = valid_seg[col].isna().sum()
    print(f'  {col:25s}: {n_nan:5d} NaN ({n_nan/len(valid_seg)*100:.1f}%)')

NaN rates in crla_school_timepoints.parquet (valid schools only):

  ordinal_mean             :     0 NaN (0.0%)
  ordinal_sd               :     0 NaN (0.0%)
  ordinal_skew             :  1754 NaN (0.9%)
  ordinal_kurt             :  1754 NaN (0.9%)
  bimodality_coef          :  1754 NaN (0.9%)
  pct_gl                   :     0 NaN (0.0%)

NaN rates in crla_school_segments.parquet (valid segments only):

  delta_mean               :     0 NaN (0.0%)
  delta_sd                 :     0 NaN (0.0%)
  delta_skew               :  2691 NaN (1.3%)
  emd_mean                 :    12 NaN (0.0%)


---
## 9. Known Data Quality Issues and Decisions

This section explicitly documents issues found in the bronze data and how they were handled.

In [28]:
print('ISSUE 1: G3 MT columns missing in EoSY and MoSY 2025-26')
print('  Root cause: G3 mother tongue assessment was not conducted at those timepoints.')
print('  Handling: harmonize_columns() fills with NaN. Pipeline handles absent groups gracefully.')
print(f'  Affected canonical columns: {[c for c in CANONICAL_GRADE_COLUMNS if "G3 MT" in c]}')
print()
for key in [('2025-26','EoSY'), ('2025-26','MoSY')]:
    df = pd.read_parquet(PROJECT_ROOT / f'data/silver/crla/{key[0]}_{key[1]}.parquet')
    g3mt = [c for c in df.columns if 'G3 MT' in c]
    if g3mt:
        nan_pct = df[g3mt[0]].isna().sum() / len(df) * 100
        print(f'  {key[1]} {key[0]}: {nan_pct:.1f}% NaN in G3 MT columns')

ISSUE 1: G3 MT columns missing in EoSY and MoSY 2025-26
  Root cause: G3 mother tongue assessment was not conducted at those timepoints.
  Handling: harmonize_columns() fills with NaN. Pipeline handles absent groups gracefully.
  Affected canonical columns: ['G3 MT Lower Emergent', 'G3 MT Higher Emergent', 'G3 MT Developing', 'G3 MT Transitioning', 'G3 MT Grade Level']



  EoSY 2025-26: 100.0% NaN in G3 MT columns


  MoSY 2025-26: 100.0% NaN in G3 MT columns


In [29]:
print('ISSUE 2: Some schools appear in one timepoint but not another (cohort dropout)')
print()
counts_by_tp = {}
for key in sorted(df_all.keys()):
    counts_by_tp[key] = set(df_all[key]['School ID'].astype(str).str.strip())

keys = sorted(counts_by_tp.keys())
for i, k in enumerate(keys):
    n = len(counts_by_tp[k])
    print(f'  {k[1]} {k[0]}: {n:,} schools')

all_ids = set.union(*counts_by_tp.values())
only_one = sum(1 for sid in all_ids if sum(sid in s for s in counts_by_tp.values()) == 1)
print(f'\nSchools appearing in exactly ONE timepoint: {only_one:,}')
print(f'Schools appearing in ALL {len(keys)} timepoints: ',
      sum(1 for sid in all_ids if all(sid in s for s in counts_by_tp.values())))

ISSUE 2: Some schools appear in one timepoint but not another (cohort dropout)

  BoSY 2024-25: 35,280 schools
  EoSY 2024-25: 37,045 schools
  BoSY 2025-26: 38,983 schools
  EoSY 2025-26: 38,743 schools
  MoSY 2025-26: 38,297 schools



Schools appearing in exactly ONE timepoint: 440


Schools appearing in ALL 5 timepoints:  34694


In [30]:
print('ISSUE 3: Zero-row or near-zero schools')
print('  Schools where all 30 grade columns are 0 or NaN after harmonization.')
print()
for key, df in df_all.items():
    clean = _clean_raw_to_numeric(df)
    grade_sums = clean[CANONICAL_GRADE_COLUMNS].sum(axis=1)
    zero_schools = (grade_sums == 0).sum()
    print(f'  {key[1]} {key[0]}: {zero_schools} schools with all-zero grade columns')

ISSUE 3: Zero-row or near-zero schools
  Schools where all 30 grade columns are 0 or NaN after harmonization.



  BoSY 2024-25: 0 schools with all-zero grade columns


  BoSY 2025-26: 1 schools with all-zero grade columns


  EoSY 2024-25: 3 schools with all-zero grade columns


  EoSY 2025-26: 0 schools with all-zero grade columns


  MoSY 2025-26: 0 schools with all-zero grade columns


In [31]:
print('ISSUE 4: total_assessed vs sum-derived count discrepancy')
print('  total_assessed comes from the dashboard"s "Total Assessed" column.')
print('  The fallback (when that column is absent) derives from per-group sums.')
print('  We verify the two agree for timepoints where the dashboard column is present.')
print()
from preprocessing import _total_assessed_cache, _grade_totals_cache

key = ('2024-25', 'BoSY')
pct = percentages[key]  # recomputes from raw, populating cache
from preprocessing import get_total_assessed
cached_total = get_total_assessed(*key)

# Re-derive from silver raw counts
df_s = pd.read_parquet(PROJECT_ROOT / f'data/silver/crla/{key[0]}_{key[1]}.parquet')
g1_sum = df_s[_get_group_columns('G1', None)].sum(axis=1)
g2 = df_s[_get_group_columns('G2','MT')].sum(axis=1).where(
    df_s[_get_group_columns('G2','MT')].sum(axis=1) > 0,
    df_s[_get_group_columns('G2','Fil')].sum(axis=1))
g3 = df_s[_get_group_columns('G3','MT')].sum(axis=1)
g3 = g3.where(g3 > 0, df_s[_get_group_columns('G3','Fil')].sum(axis=1))
g3 = g3.where(g3 > 0, df_s[_get_group_columns('G3','Eng')].sum(axis=1))
derived = (g1_sum.fillna(0) + g2.fillna(0) + g3.fillna(0))

common_schools = cached_total.index.intersection(derived.index)
cached_aligned = cached_total.reindex(common_schools)
derived_aligned = derived.reindex(common_schools)

diff = (cached_aligned - derived_aligned).abs()
print(f'  Max absolute difference (dashboard vs derived): {diff.max():.0f}')
print(f'  Schools with non-zero difference: {(diff > 0).sum()}')
print(f'  Reason: dashboard Total Assessed counts unique students;')
print(f'  derived sum can slightly differ if a student was assessed in multiple groups')

ISSUE 4: total_assessed vs sum-derived count discrepancy
  total_assessed comes from the dashboard"s "Total Assessed" column.
  The fallback (when that column is absent) derives from per-group sums.
  We verify the two agree for timepoints where the dashboard column is present.

  Max absolute difference (dashboard vs derived): 28
  Schools with non-zero difference: 2
  Reason: dashboard Total Assessed counts unique students;
  derived sum can slightly differ if a student was assessed in multiple groups


---
## Summary

| Decision | Rationale |
|---|---|
| Drop grade-level Total columns | Vary in naming across years; derivable from profile counts |
| Fill G3 MT with NaN for EoSY/MoSY 2025-26 | Assessment not conducted; preserves correct group coverage |
| Use dashboard Total Assessed (not derived) | Counts unique students; derived sum inflates multi-group students |
| `valid_strict` threshold: ≥4 groups, ≥15 per group | Balances coverage and statistical reliability |
| BC as relative indicator, not absolute classifier | On a 5-point ordinal scale, even uniform distributions exceed the 0.555 threshold |
| EMD computed per grade-language group, then averaged | Avoids mixing grade-level distributional shifts |
| Gold parquets gitignored | Large binary artifacts; regenerated via build_silver.py + build_gold.py |